# Example 7.3 / 1: Derive M from Bounds

Companion notebook for **7.3 Big-M Formulations** in the AMPL lab Documentation, section "Derive M from Bounds". It runs the page's example from top to bottom: the model, the amplpy way (data.py, run.py), then the pure-AMPL way (data.dat, model.run). Solver: HiGHS.

## Install Dependencies
💡 TIP: Visit [portal.ampl.com](https://portal.ampl.com/user/ampl/license/list) or [dev.ampl.com](https://dev.ampl.com/ampl/install.html) for AMPL setup on Colab, Docker, Python, Windows, Linux, macOS

In [ ]:
# Install dependencies
%pip install -q amplpy

# Google Colab & Kaggle integration
from amplpy import AMPL, ampl_notebook
ampl = ampl_notebook(
    modules=["highs"],      # modules to install
    license_uuid="default") # license to use

## Derive M from Bounds

For `body = sum {i in I} a[i] * x[i]`, with finite bounds `L[i] <= x[i] <= U[i]`, the smallest
valid value is

$$
M = \max_{L \le x \le U} \left(\sum_i a_i x_i - \text{limit}\right).
$$

For a box of independent bounds, use `U[i]` when `a[i]` is positive and `L[i]` when it is negative.
If the result is nonpositive, the constraint never needs relaxing. For a single variable with
upper bound `U`, this is simply `M = U - limit`.

In the steel model, suppose the finishing line runs in one of two modes: either bands are limited
to 3,000 tons or coils to 1,000 tons. `Apply[p] = 1` selects the limit that holds. Each
`Make[p]` is bounded by `market[p]`, so the derived constant is `market[p] - limit[p]`, written
as the default of `M`:

### model.mod
The model, written to a file that both ways of running it read.

In [2]:
%%writefile model.mod
set PROD;                         # products
param rate {PROD} > 0;            # tons produced per hour
param profit {PROD};              # profit per ton
param market {PROD} >= 0;         # most that can be sold, in tons
param avail >= 0;                 # hours available
set LIMITED within PROD;          # products with a mode-dependent limit
param limit {LIMITED} >= 0;       # tons allowed when the limit applies
param M {p in LIMITED} >= 0 default market[p] - limit[p];  # derived from the bound on Make

var Make {p in PROD} >= 0, <= market[p];  # tons produced
var Apply {LIMITED} binary;               # 1 if the product's limit holds

maximize TotalProfit: sum {p in PROD} profit[p] * Make[p];
subject to Time: sum {p in PROD} (1 / rate[p]) * Make[p] <= avail;  # hours available
subject to OneMode: sum {p in LIMITED} Apply[p] = 1;                # exactly one limit holds
# the limit is enforced when Apply[p] = 1 and relaxed by M[p] otherwise
subject to Conditional {p in LIMITED}:
    Make[p] <= limit[p] + M[p] * (1 - Apply[p]);

Writing model.mod


### amplpy: data.py and run.py

`ampl` is the `AMPL` object created above. Each run starts from a fresh session that has read `model.mod`; then `data.py` sets the data from Python and `run.py` solves and reads the results.

In [3]:
# A fresh session that has read model.mod, as data.py and run.py assume
ampl.reset()
ampl.eval("reset options;")  # reset keeps the options
ampl.read("model.mod")

In [4]:
# data.py
ampl.set["PROD"] = ["bands", "coils", "plate"]
ampl.param["rate"] = {"bands": 200, "coils": 140, "plate": 160}
ampl.param["profit"] = {"bands": 25, "coils": 30, "plate": 29}
ampl.param["market"] = {"bands": 6000, "coils": 4000, "plate": 3500}
ampl.param["avail"] = 40
ampl.set["LIMITED"] = ["bands", "coils"]
ampl.param["limit"] = {"bands": 3000, "coils": 1000}

In [5]:
# run.py
ampl.option["solver_msg"] = 0
ampl.solve(solver="highs")
print("M:", ampl.param["M"].to_dict())
print(ampl.solve_result, ampl.obj["TotalProfit"].value())
print(ampl.get_data("Make").to_dict(), ampl.var["Apply"].to_dict())

M: {'bands': 3000, 'coils': 3000}
solved 196400.0
{'bands': 6000, 'coils': 0, 'plate': 1600} {'bands': 0, 'coils': 1}


### Another way in amplpy

The model applies the coils limit (`Apply["coils"] = 1`), which costs nothing because coils are
not worth making, and lifts the bands row by `M["bands"] = 3000` to the full market of 6,000 tons.

A guessed constant can be too small. With the same `model.mod` and `data.py`, a "large enough"
`M` of 1,000 for both rows still caps bands at 3,000 + 1,000 = 4,000 tons when their limit is
off:

In [6]:
# A fresh session that has read model.mod, as data.py and run.py assume
ampl.reset()
ampl.eval("reset options;")  # reset keeps the options
ampl.read("model.mod")

In [7]:
# data.py
ampl.set["PROD"] = ["bands", "coils", "plate"]
ampl.param["rate"] = {"bands": 200, "coils": 140, "plate": 160}
ampl.param["profit"] = {"bands": 25, "coils": 30, "plate": 29}
ampl.param["market"] = {"bands": 6000, "coils": 4000, "plate": 3500}
ampl.param["avail"] = 40
ampl.set["LIMITED"] = ["bands", "coils"]
ampl.param["limit"] = {"bands": 3000, "coils": 1000}

In [8]:
# run.py
ampl.option["solver_msg"] = 0
ampl.param["M"] = {"bands": 1000, "coils": 1000}
ampl.solve(solver="highs")
print(ampl.solve_result, ampl.obj["TotalProfit"].value())
print(ampl.get_data("Make").to_dict(), ampl.var["Apply"].to_dict())

solved 192800.0
{'bands': 4000, 'coils': 0, 'plate': 3200} {'bands': 0, 'coils': 1}


### Pure AMPL: data.dat and model.run

> **Pitfall: `M = 1000` is not a derivation**
> The solver still reports `solved`, but the profit drops from 196,400 to 192,800: nothing in the
> status reveals that the constant cut off the true optimum. A value that is far too large, such
> as `10^6`, keeps the optimum but weakens the relaxation, as the next section explains.

The same in pure AMPL, with the same `model.mod`:

In [9]:
%%writefile data.dat
param: PROD: rate profit market :=
  bands  200  25  6000
  coils  140  30  4000
  plate  160  29  3500 ;
param avail := 40;
set LIMITED := bands coils;
param limit := bands 3000  coils 1000;

Writing data.dat


In [10]:
%%ampl_eval
reset;
reset options;
model model.mod;
data data.dat;

In [11]:
%%ampl_eval
# model.run
option solver highs;
option solver_msg 0;
solve;
display M, TotalProfit, Make, Apply;

M [*] :=
bands  3000
coils  3000
;

TotalProfit = 196400

:       Make Apply    :=
bands   6000    0
coils      0    1
plate   1600    .
;

